# Task 03 – Data Preprocessing and Feature Engineering
### SmartCare Hospital AI Dataset | Option C – Disease Risk Classification
### CCS3440 Artificial Intelligence Coursework

**Target variable:** `disease_risk_level` (Low / Medium / High)

This notebook follows a professional, section-wise preprocessing pipeline. Every
transformation is:
1. Applied only after being justified,
2. Fitted on training data only where applicable (to avoid data leakage), and
3. Logged with before/after shape and row counts so the workflow is auditable.

**Pipeline order:** Setup → Missing Values → Duplicates → Outliers → Data Cleaning →
Encoding → Scaling → Feature Selection → Feature Engineering → Train/Test Split → Save.


## Section 1 — Setup and Data Load

Import libraries and load the raw dataset. We keep a copy of the original
dataframe (`df_raw`) untouched throughout, and perform all transformations on
a working copy (`df`), so every step can be verified against the original at
any point (useful for the viva).


In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RANDOM_STATE = 42

# Load raw data
df_raw = pd.read_csv('../data/raw/smartcare_ai_dataset_1000.csv')
df = df_raw.copy()

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Dataset shape: (1000, 33)
Columns: ['record_id', 'patient_id', 'age', 'gender', 'blood_group', 'department', 'diagnosis', 'appointment_date', 'waiting_days', 'previous_appointments', 'missed_previous_appointments', 'appointment_status', 'admitted', 'room_type', 'length_of_stay_days', 'previous_admissions', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mg_dl', 'cholesterol_mg_dl', 'bmi', 'lab_tests_count', 'treatments_count', 'consultation_fee_lkr', 'room_charge_lkr', 'lab_charge_lkr', 'medicine_charge_lkr', 'total_bill_lkr', 'payment_status', 'payment_method', 'no_show', 'readmitted_30_days', 'disease_risk_level']


,record_id,patient_id,age,gender,blood_group,department,diagnosis,appointment_date,waiting_days,previous_appointments,missed_previous_appointments,appointment_status,admitted,room_type,length_of_stay_days,previous_admissions,systolic_bp,diastolic_bp,blood_sugar_mg_dl,cholesterol_mg_dl,bmi,lab_tests_count,treatments_count,consultation_fee_lkr,room_charge_lkr,lab_charge_lkr,medicine_charge_lkr,total_bill_lkr,payment_status,payment_method,no_show,readmitted_30_days,disease_risk_level
0,1,P10001,53,Male,A-,General Medicine,Migraine,2025-04-10,10,1,0,Completed,0,NaN,0,1,127,75,117,211,26.1,0,3,2000,0,0,11596,13596,Paid,Insurance,0,0,High
1,2,P10002,26,Male,B-,General Medicine,Diabetes,2025-05-15,2,3,1,Completed,0,NaN,0,0,130,73,136,173,32.8,0,1,2000,0,0,3652,5652,Paid,Insurance,0,0,Medium
2,3,P10003,22,Male,B+,Orthopedics,Back Pain,2025-07-09,22,7,1,No-Show,0,NaN,0,1,141,64,90,176,29.4,1,0,2500,0,1200,2562,6262,Unpaid,Insurance,1,0,Medium
3,4,P10004,44,Female,AB-,Cardiology,Asthma,2025-10-16,16,1,0,Completed,0,NaN,0,0,124,82,126,189,24.9,2,1,2000,0,5000,10262,17262,Paid,Online,0,0,Medium
4,5,P10005,51,Female,O+,Neurology,Hypertension,2025-12-18,12,4,0,Scheduled,0,NaN,0,1,119,81,65,195,27.0,2,0,4000,0,6000,10414,20414,Paid,Cash,0,0,Medium


In [5]:
# Quick structural check before any changes
print("Data types:")
print(df.dtypes)
print("\nTarget distribution (disease_risk_level):")
print(df['disease_risk_level'].value_counts())
print(df['disease_risk_level'].value_counts(normalize=True).round(3) * 100)

Data types:
record_id                         int64
patient_id                       object
age                               int64
gender                           object
blood_group                      object
department                       object
diagnosis                        object
appointment_date                 object
waiting_days                      int64
previous_appointments             int64
missed_previous_appointments      int64
appointment_status               object
admitted                          int64
room_type                        object
length_of_stay_days               int64
previous_admissions               int64
systolic_bp                       int64
diastolic_bp                      int64
blood_sugar_mg_dl                 int64
cholesterol_mg_dl                 int64
bmi                             float64
lab_tests_count                   int64
treatments_count                  int64
consultation_fee_lkr              int64
room_charge_lkr             

## Section 2 — Missing Value Handling

**Finding:** Only `room_type` has missing values (906/1000 rows, 90.6%).

**Why a single imputation strategy is wrong here:** the missingness is not
uniform in cause.
- For **non-admitted patients** (`admitted == 0`), `room_type` is missing
  because no room was ever assigned — this is **Missing Not At Random (MNAR)**
  by design, not an error.
- For **admitted patients** (`admitted == 1`) with a blank `room_type`, this
  is genuine missing data — **Missing At Random (MAR)**.

**Decision:** Split the imputation logic by `admitted` status rather than
applying one blanket rule. Non-admitted patients get an explicit
`"Not Admitted"` category (preserves information, avoids treating 90.6% of
the dataset as "unknown"). Admitted-but-missing patients are imputed with
the mode room type among admitted patients, which is the least biased
single-value estimate available without additional external data.


In [6]:
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Diagnostic: confirm the missingness pattern is tied to admission status
print("\nroom_type missing, broken down by admitted status:")
print(df.groupby('admitted')['room_type'].apply(lambda x: x.isnull().sum()))

Missing values per column:
room_type    906
dtype: int64

room_type missing, broken down by admitted status:
admitted
0    670
1    236
Name: room_type, dtype: int64


In [7]:
# Step 1a: Non-admitted patients -> explicit "Not Admitted" category (MNAR, not an error)
mask_not_admitted = (df['admitted'] == 0) & (df['room_type'].isnull())
df.loc[mask_not_admitted, 'room_type'] = 'Not Admitted'
print(f"Rows set to 'Not Admitted': {mask_not_admitted.sum()}")

# Step 1b: Admitted patients with missing room_type -> impute with mode (MAR, true gap)
admitted_mode = df.loc[(df['admitted'] == 1) & (df['room_type'].notnull()), 'room_type'].mode()[0]
mask_admitted_missing = (df['admitted'] == 1) & (df['room_type'].isnull())
df.loc[mask_admitted_missing, 'room_type'] = admitted_mode
print(f"Rows imputed with mode ('{admitted_mode}'): {mask_admitted_missing.sum()}")

# Verify no missing values remain
print(f"\nRemaining missing values:\n{df.isnull().sum().sum()} total")
assert df['room_type'].isnull().sum() == 0, "room_type still has nulls!"
print("All missing values resolved.")

Rows set to 'Not Admitted': 670
Rows imputed with mode ('General Ward'): 236

Remaining missing values:
0 total
All missing values resolved.


## Section 3 — Duplicate Record Detection

Even though the coursework brief requires this check regardless of outcome,
it must still be explicitly run and documented — not assumed. We check three
levels: full-row duplicates, duplicate `record_id`, and duplicate `patient_id`
(patients may legitimately appear more than once across different visits,
so a duplicate `patient_id` alone is *not* an error — only full-row or
`record_id` duplicates would be).


In [8]:
full_duplicates = df.duplicated().sum()
duplicate_record_ids = df['record_id'].duplicated().sum()
duplicate_patient_ids = df['patient_id'].duplicated().sum()  # informational only

print(f"Full-row duplicates: {full_duplicates}")
print(f"Duplicate record_id values: {duplicate_record_ids}")
print(f"Repeated patient_id values (expected, not an error — same patient, multiple visits): {duplicate_patient_ids}")

if full_duplicates > 0:
    df = df.drop_duplicates()
    print(f"Dropped {full_duplicates} duplicate rows. New shape: {df.shape}")
else:
    print("No full-row or record_id duplicates found — no action required.")

Full-row duplicates: 0
Duplicate record_id values: 0
Repeated patient_id values (expected, not an error — same patient, multiple visits): 0
No full-row or record_id duplicates found — no action required.


## Section 3 — Outlier Identification

**Approach:** IQR method applied to continuous clinical variables that will
be used as model predictors.

**Key decision — outliers are identified but NOT removed or capped for
clinical vitals.** In a disease-risk classification problem, extreme values
(e.g. very high cholesterol, very high BMI) are very likely the *exact
signal* driving the "High" risk label, not noise. Removing them would strip
the model of the information it most needs. Instead, we flag them and
cross-check their relationship to the target as evidence for this decision.

Billing outliers are checked separately for legitimacy (not used as
predictors for Option C — see Feature Selection).


In [9]:
def iqr_outlier_report(series, col_name):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    print(f"{col_name:22s} | bounds=({lower:.1f}, {upper:.1f}) | outliers={len(outliers)}")
    return outliers.index

clinical_cols = ['age', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mg_dl',
                  'cholesterol_mg_dl', 'bmi']

outlier_indices = {}
for col in clinical_cols:
    outlier_indices[col] = iqr_outlier_report(df[col], col)

age                    | bounds=(-3.0, 93.0) | outliers=0
systolic_bp            | bounds=(84.0, 172.0) | outliers=3
diastolic_bp           | bounds=(51.0, 107.0) | outliers=3
blood_sugar_mg_dl      | bounds=(44.5, 184.5) | outliers=10
cholesterol_mg_dl      | bounds=(105.0, 305.0) | outliers=6
bmi                    | bounds=(14.3, 36.7) | outliers=9


In [10]:
# Cross-check: do outliers skew toward the "High" risk class?
# This justifies KEEPING them rather than removing them.
for col in ['bmi', 'cholesterol_mg_dl']:
    idx = outlier_indices[col]
    if len(idx) > 0:
        print(f"\n{col} outlier rows — disease_risk_level distribution:")
        print(df.loc[idx, 'disease_risk_level'].value_counts())


bmi outlier rows — disease_risk_level distribution:
disease_risk_level
High      5
Medium    3
Low       1
Name: count, dtype: int64

cholesterol_mg_dl outlier rows — disease_risk_level distribution:
disease_risk_level
High      3
Medium    2
Low       1
Name: count, dtype: int64


In [11]:
# Billing outlier legitimacy check (not used as a predictor, but verified for completeness)
paid_rows = df[df['room_charge_lkr'] > 0].copy()
paid_rows['rate_per_day'] = paid_rows['room_charge_lkr'] / paid_rows['length_of_stay_days']
print("Room charge rate per day, by room_type (should be constant if legitimate):")
print(paid_rows.groupby('room_type')['rate_per_day'].describe()[['mean', 'std', 'min', 'max']])
print("\n Std = 0 confirms room charges follow a fixed daily rate — not data errors, so no cleaning needed.")

Room charge rate per day, by room_type (should be constant if legitimate):
                 mean  std      min      max
room_type                                   
General Ward   3500.0  0.0   3500.0   3500.0
ICU           25000.0  0.0  25000.0  25000.0
Private Room   9000.0  0.0   9000.0   9000.0

 Std = 0 confirms room charges follow a fixed daily rate — not data errors, so no cleaning needed.


## Section 4 — Data Cleaning

Three concrete logical/physiological errors were identified during Task 02
and are corrected here, each with an explicit justification. No rows are
dropped — corrections are applied in place to preserve the full dataset.

| Issue | Rows | Action | Justification |
|---|---|---|---|
| `missed_previous_appointments > previous_appointments` | 16 | Cap at `previous_appointments` | A patient cannot miss more appointments than they had |
| `systolic_bp <= diastolic_bp` | 3 | Swap the two values | More parsimonious than dropping — treated as a data-entry transposition |
| `treatments_count == 0` but `medicine_charge_lkr > 0` | 212 | No correction — documented only | Billing inconsistency unrelated to the clinical predictors used for Option C; fixing it is out of scope and would not affect the disease-risk model |


In [12]:
# Issue 1: missed_previous_appointments cannot exceed previous_appointments
mask_bad_missed = df['missed_previous_appointments'] > df['previous_appointments']
print(f"Rows with impossible missed-appointment count: {mask_bad_missed.sum()}")

df.loc[mask_bad_missed, 'missed_previous_appointments'] = df.loc[mask_bad_missed, 'previous_appointments']

assert (df['missed_previous_appointments'] <= df['previous_appointments']).all()
print("Fixed: missed_previous_appointments capped at previous_appointments.")

Rows with impossible missed-appointment count: 16
Fixed: missed_previous_appointments capped at previous_appointments.


In [13]:
# Issue 2: systolic_bp must exceed diastolic_bp (physiological validity)
mask_bad_bp = df['systolic_bp'] <= df['diastolic_bp']
print(f"Rows with invalid BP (systolic <= diastolic): {mask_bad_bp.sum()}")
print(df.loc[mask_bad_bp, ['record_id', 'systolic_bp', 'diastolic_bp']])

# Swap the two values for affected rows (treat as transposition error)
df.loc[mask_bad_bp, ['systolic_bp', 'diastolic_bp']] = \
    df.loc[mask_bad_bp, ['diastolic_bp', 'systolic_bp']].values

assert (df['systolic_bp'] > df['diastolic_bp']).all()
print("Fixed: systolic/diastolic values swapped for affected rows. No rows dropped.")

Rows with invalid BP (systolic <= diastolic): 3
     record_id  systolic_bp  diastolic_bp
99         100           85            86
331        332           98           103
374        375           94            97
Fixed: systolic/diastolic values swapped for affected rows. No rows dropped.


In [14]:
# Issue 3: treatments_count == 0 but medicine_charge_lkr > 0 -> documented, not corrected
mask_billing_inconsistency = (df['treatments_count'] == 0) & (df['medicine_charge_lkr'] > 0)
print(f"Billing inconsistency rows (documented only, not modified): {mask_billing_inconsistency.sum()}")
print("Rationale: unrelated to clinical predictors for disease_risk_level — out of scope for Option C.")

print(f"\n--- Cleaning summary ---")
print(f"Final row count after cleaning: {len(df)} (started with {len(df_raw)}, 0 rows dropped)")

Billing inconsistency rows (documented only, not modified): 212
Rationale: unrelated to clinical predictors for disease_risk_level — out of scope for Option C.

--- Cleaning summary ---
Final row count after cleaning: 1000 (started with 1000, 0 rows dropped)


## Section 5 — Feature Encoding

- **Nominal categorical predictors** (`gender`, `blood_group`, `department`,
  `diagnosis`, `room_type`, `payment_method`, `payment_status`,
  `appointment_status`): **one-hot encoded**. These categories have no
  natural order — label encoding would falsely imply a ranking, which would
  mislead distance-based/linear models (Logistic Regression, KNN, SVM).
- **Target variable** `disease_risk_level`: **ordinal encoded**
  (Low=0, Medium=1, High=2) since the classes have a genuine order. The
  original string labels are preserved in a separate column for readable
  confusion matrices and report visuals.


In [15]:
# Preserve original readable target labels for reporting
df['disease_risk_level_label'] = df['disease_risk_level']

# Ordinal encode the target (natural order: Low < Medium < High)
risk_order = {'Low': 0, 'Medium': 1, 'High': 2}
df['disease_risk_level'] = df['disease_risk_level_label'].map(risk_order)

print("Target encoding check:")
print(df[['disease_risk_level_label', 'disease_risk_level']].drop_duplicates())

Target encoding check:
   disease_risk_level_label  disease_risk_level
0                      High                   2
1                    Medium                   1
12                      Low                   0


In [16]:
# One-hot encode nominal predictor columns
nominal_cols = ['gender', 'blood_group', 'department', 'diagnosis', 'room_type',
                'payment_method', 'payment_status', 'appointment_status']

df_encoded = pd.get_dummies(df, columns=nominal_cols, drop_first=False)

print(f"Shape before encoding: {df.shape}")
print(f"Shape after encoding:  {df_encoded.shape}")
print(f"\nNew one-hot columns created: {df_encoded.shape[1] - df.shape[1] + len(nominal_cols)}")


Shape before encoding: (1000, 34)
Shape after encoding:  (1000, 68)

New one-hot columns created: 42


## Section 6 — Feature Scaling

`StandardScaler` (z-score standardization) is applied to continuous numeric
predictors. This is required for distance-based/gradient-based models
(Logistic Regression, KNN, SVM) which are sensitive to feature magnitude —
without scaling, `cholesterol_mg_dl` (range ~100–330) would dominate `bmi`
(range ~14–39) purely due to scale, not actual importance.

**Critical detail:** the scaler is fit **only on the training set** after
the train/test split (Section 9), then used to transform both sets. Fitting
on the full dataset before splitting would leak test-set statistics into
training — a data leakage error. This cell defines the scaling step; it is
executed after the split in Section 9.


In [17]:
numeric_cols_to_scale = ['age', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mg_dl',
                          'cholesterol_mg_dl', 'bmi', 'previous_admissions']

# Scaler object created here; fit/transform is deferred to Section 9 (post train/test split)
scaler = StandardScaler()
print("StandardScaler initialized for columns:", numeric_cols_to_scale)
print("NOTE: scaler.fit() is called on X_train only, in Section 9, to avoid data leakage.")

StandardScaler initialized for columns: ['age', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mg_dl', 'cholesterol_mg_dl', 'bmi', 'previous_admissions']
NOTE: scaler.fit() is called on X_train only, in Section 9, to avoid data leakage.


## Section 7 — Feature Selection

Based on correlation analysis against `disease_risk_level` (computed during
Task 02 EDA):

| Decision | Columns | Reason |
|---|---|---|
| **Drop — identifiers** | `record_id`, `patient_id` | No predictive value, risk of overfitting |
| **Drop — other targets** | `no_show`, `readmitted_30_days` | These are the target variables for Options A and B; using one AI target to predict another is poor methodology |
| **Drop — near-zero correlation** | `total_bill_lkr`, `lab_tests_count`, `treatments_count`, `waiting_days`, `consultation_fee_lkr`, `room_charge_lkr`, `lab_charge_lkr`, `medicine_charge_lkr` | r ≈ 0.00–0.05 with target; billing/operational fields not clinically linked to risk level |
| **Drop — administrative** | `appointment_date`, `disease_risk_level_label` (kept aside for reporting only) | Not usable as a numeric/categorical model feature as-is |
| **Keep — strong predictors** | `age`, `systolic_bp`, `diastolic_bp`, `blood_sugar_mg_dl`, `cholesterol_mg_dl`, `bmi`, `previous_admissions` | r = 0.17–0.54 with target |
| **Keep with caveat** | one-hot encoded `department`, `diagnosis` | Included for completeness; EDA showed no coherent clinical relationship between them, so Task 07 (SHAP/feature importance) is expected to confirm their low contribution rather than assuming it upfront |


In [18]:
# Confirm the correlation basis for these decisions
numeric_for_corr = ['age', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mg_dl',
                     'cholesterol_mg_dl', 'bmi', 'waiting_days', 'previous_appointments',
                     'missed_previous_appointments', 'previous_admissions',
                     'lab_tests_count', 'treatments_count', 'total_bill_lkr',
                     'disease_risk_level']

corr_with_target = df[numeric_for_corr].corr()['disease_risk_level'].sort_values(ascending=False)
print("Correlation of numeric features with disease_risk_level:")
print(corr_with_target)

Correlation of numeric features with disease_risk_level:
disease_risk_level              1.000000
age                             0.537582
blood_sugar_mg_dl               0.474440
cholesterol_mg_dl               0.446766
bmi                             0.359904
systolic_bp                     0.310182
previous_admissions             0.171077
diastolic_bp                    0.085260
previous_appointments           0.049035
missed_previous_appointments    0.027006
waiting_days                    0.018815
total_bill_lkr                  0.011739
lab_tests_count                -0.001361
treatments_count               -0.002355
Name: disease_risk_level, dtype: float64


In [19]:
# Apply feature selection
drop_cols = [
    'record_id', 'patient_id',                      # identifiers
    'no_show', 'readmitted_30_days',                 # other AI targets - leakage/scope risk
    'total_bill_lkr', 'lab_tests_count', 'treatments_count', 'waiting_days',
    'consultation_fee_lkr', 'room_charge_lkr', 'lab_charge_lkr', 'medicine_charge_lkr',
    'appointment_date',
    'disease_risk_level_label',                       # kept aside for reporting, not modeling
]

df_selected = df_encoded.drop(columns=drop_cols)
print(f"Shape before selection: {df_encoded.shape}")
print(f"Shape after selection:  {df_selected.shape}")
print(f"\nRemaining columns:\n{list(df_selected.columns)}")

Shape before selection: (1000, 68)
Shape after selection:  (1000, 54)

Remaining columns:
['age', 'previous_appointments', 'missed_previous_appointments', 'admitted', 'length_of_stay_days', 'previous_admissions', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mg_dl', 'cholesterol_mg_dl', 'bmi', 'disease_risk_level', 'gender_Female', 'gender_Male', 'blood_group_A+', 'blood_group_A-', 'blood_group_AB+', 'blood_group_AB-', 'blood_group_B+', 'blood_group_B-', 'blood_group_O+', 'blood_group_O-', 'department_Cardiology', 'department_General Medicine', 'department_Laboratory Services', 'department_Neurology', 'department_Orthopedics', 'department_Pediatrics', 'department_Radiology', 'diagnosis_Asthma', 'diagnosis_Back Pain', 'diagnosis_Chest Pain', 'diagnosis_Diabetes', 'diagnosis_Fever', 'diagnosis_Fracture', 'diagnosis_Hypertension', 'diagnosis_Kidney Infection', 'diagnosis_Migraine', 'diagnosis_Pneumonia', 'room_type_General Ward', 'room_type_ICU', 'room_type_Not Admitted', 'room_type_Private

## Section 8 — Feature Engineering

Raw vitals alone do not cleanly separate the three risk classes (ranges
overlap heavily across Low/Medium/High per Task 02 EDA). Converting
continuous vitals into clinically meaningful bands using standard medical
thresholds captures non-linear threshold effects that linear models cannot
otherwise learn directly, and improves interpretability for the
Explainable AI stage (Task 07).

| New feature | Source | Bands used |
|---|---|---|
| `bp_category` | `systolic_bp`, `diastolic_bp` | Normal / Elevated / Hypertensive (AHA guidelines) |
| `bmi_category` | `bmi` | Underweight / Normal / Overweight / Obese (WHO bands) |
| `blood_sugar_category` | `blood_sugar_mg_dl` | Normal / Prediabetic / Diabetic |
| `age_group` | `age` | Child / Young Adult / Adult / Senior |
| `health_burden_score` | `previous_admissions` + `previous_appointments` | Composite numeric score |

These engineered features are computed on `df` (pre-encoding) then merged
back so they can be one-hot encoded alongside the other categoricals.


In [20]:
def bp_category(row):
    s, d = row['systolic_bp'], row['diastolic_bp']
    if s < 120 and d < 80:
        return 'Normal'
    elif s < 130 and d < 80:
        return 'Elevated'
    else:
        return 'Hypertensive'

def bmi_category(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif bmi < 25:
        return 'Normal'
    elif bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'

def blood_sugar_category(bs):
    if bs < 100:
        return 'Normal'
    elif bs < 126:
        return 'Prediabetic'
    else:
        return 'Diabetic'

def age_group(age):
    if age < 13:
        return 'Child'
    elif age < 30:
        return 'Young Adult'
    elif age < 60:
        return 'Adult'
    else:
        return 'Senior'

df['bp_category'] = df.apply(bp_category, axis=1)
df['bmi_category'] = df['bmi'].apply(bmi_category)
df['blood_sugar_category'] = df['blood_sugar_mg_dl'].apply(blood_sugar_category)
df['age_group'] = df['age'].apply(age_group)
df['health_burden_score'] = df['previous_admissions'] + df['previous_appointments']

print("Engineered feature previews:")
print(df[['bp_category', 'bmi_category', 'blood_sugar_category', 'age_group',
          'health_burden_score']].head())

print("\nDistribution of engineered categorical features:")
for col in ['bp_category', 'bmi_category', 'blood_sugar_category', 'age_group']:
    print(f"\n{col}:")
    print(df[col].value_counts())

Engineered feature previews:
    bp_category bmi_category blood_sugar_category    age_group  health_burden_score
0      Elevated   Overweight          Prediabetic        Adult                    2
1  Hypertensive        Obese             Diabetic  Young Adult                    3
2  Hypertensive   Overweight               Normal  Young Adult                    8
3  Hypertensive       Normal             Diabetic        Adult                    1
4  Hypertensive   Overweight               Normal        Adult                    5

Distribution of engineered categorical features:

bp_category:
bp_category
Hypertensive    710
Normal          168
Elevated        122
Name: count, dtype: int64

bmi_category:
bmi_category
Overweight     403
Normal         386
Obese          155
Underweight     56
Name: count, dtype: int64

blood_sugar_category:
blood_sugar_category
Prediabetic    389
Diabetic       323
Normal         288
Name: count, dtype: int64

age_group:
age_group
Adult          598
Senior 

In [21]:
# Merge engineered features into the selected/encoded feature set
engineered_cols = ['bp_category', 'bmi_category', 'blood_sugar_category', 'age_group',
                    'health_burden_score']

df_final = df_selected.copy()
for col in ['bp_category', 'bmi_category', 'blood_sugar_category', 'age_group']:
    dummies = pd.get_dummies(df[col], prefix=col)
    df_final = pd.concat([df_final, dummies], axis=1)

df_final['health_burden_score'] = df['health_burden_score']

print(f"Final shape after feature engineering: {df_final.shape}")
df_final.head()

Final shape after feature engineering: (1000, 69)


,age,previous_appointments,missed_previous_appointments,admitted,length_of_stay_days,previous_admissions,systolic_bp,diastolic_bp,blood_sugar_mg_dl,cholesterol_mg_dl,bmi,disease_risk_level,gender_Female,gender_Male,blood_group_A+,blood_group_A-,blood_group_AB+,blood_group_AB-,blood_group_B+,blood_group_B-,blood_group_O+,blood_group_O-,department_Cardiology,department_General Medicine,department_Laboratory Services,department_Neurology,department_Orthopedics,department_Pediatrics,department_Radiology,diagnosis_Asthma,diagnosis_Back Pain,diagnosis_Chest Pain,diagnosis_Diabetes,diagnosis_Fever,diagnosis_Fracture,diagnosis_Hypertension,diagnosis_Kidney Infection,diagnosis_Migraine,diagnosis_Pneumonia,room_type_General Ward,room_type_ICU,room_type_Not Admitted,room_type_Private Room,payment_method_Card,payment_method_Cash,payment_method_Insurance,payment_method_Online,payment_status_Paid,payment_status_Partially Paid,payment_status_Unpaid,appointment_status_Cancelled,appointment_status_Completed,appointment_status_No-Show,appointment_status_Scheduled,bp_category_Elevated,bp_category_Hypertensive,bp_category_Normal,bmi_category_Normal,bmi_category_Obese,bmi_category_Overweight,bmi_category_Underweight,blood_sugar_category_Diabetic,blood_sugar_category_Normal,blood_sugar_category_Prediabetic,age_group_Adult,age_group_Child,age_group_Senior,age_group_Young Adult,health_burden_score
0,53,1,0,0,0,1,127,75,117,211,26.1,2,False,True,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,True,False,True,False,False,False,True,False,False,True,False,False,False,False,True,False,False,False,True,True,False,False,False,2
1,26,3,1,0,0,0,130,73,136,173,32.8,1,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,True,False,False,False,True,False,False,True,False,False,True,False,False,False,False,False,True,3
2,22,7,1,0,0,1,141,64,90,176,29.4,1,False,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,True,False,False,True,False,False,True,False,False,False,True,False,False,True,False,False,False,False,True,8
3,44,1,0,0,0,0,124,82,126,189,24.9,1,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,True,False,False,False,True,False,False,False,True,False,True,False,False,False,True,False,False,True,False,False,False,1
4,51,4,0,0,0,1,119,81,65,195,27.0,1,True,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,True,False,False,True,False,False,False,False,False,True,False,True,False,False,False,True,False,False,True,False,True,False,False,False,5


## Section 9 — Class Imbalance Check, Train/Test Split, and Scaling

**Class imbalance:** Low=13.1%, Medium=46.9%, High=40.0%. A stratified split
ensures all three classes are proportionally represented in both train and
test sets, which is essential given the imbalance — a random (non-stratified)
split risks under-representing the minority "Low" class in the test set,
producing unreliable evaluation metrics in Task 06.

**Scaling is fit on the training set only** (as specified in Section 6) to
prevent data leakage — the test set must remain fully unseen during fitting.


In [22]:
X = df_final.drop(columns=['disease_risk_level'])
y = df_final['disease_risk_level']

print("Class distribution (full dataset):")
print(y.value_counts(normalize=True).round(3) * 100)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}")
print("\nClass distribution (train):")
print(y_train.value_counts(normalize=True).round(3) * 100)
print("\nClass distribution (test):")
print(y_test.value_counts(normalize=True).round(3) * 100)

Class distribution (full dataset):
disease_risk_level
1    46.9
2    40.0
0    13.1
Name: proportion, dtype: float64

Train shape: (800, 68), Test shape: (200, 68)

Class distribution (train):
disease_risk_level
1    46.9
2    40.0
0    13.1
Name: proportion, dtype: float64

Class distribution (test):
disease_risk_level
1    47.0
2    40.0
0    13.0
Name: proportion, dtype: float64


In [23]:
# Fit scaler on TRAIN ONLY, then transform both sets (no leakage)
scaler = StandardScaler()
X_train[numeric_cols_to_scale] = scaler.fit_transform(X_train[numeric_cols_to_scale])
X_test[numeric_cols_to_scale] = scaler.transform(X_test[numeric_cols_to_scale])

print("Scaling applied — fit on X_train, transform on X_train and X_test.")
X_train[numeric_cols_to_scale].describe().round(2)

Scaling applied — fit on X_train, transform on X_train and X_test.


,age,systolic_bp,diastolic_bp,blood_sugar_mg_dl,cholesterol_mg_dl,bmi,previous_admissions
count,800.00,800.00,800.00,800.00,800.00,800.00,800.00
mean,0.00,-0.00,-0.00,-0.00,0.00,0.00,0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-2.45,-2.83,-2.89,-1.88,-2.84,-2.73,-0.90
25%,-0.68,-0.69,-0.70,-0.69,-0.70,-0.65,-0.90
50%,-0.01,-0.04,-0.01,-0.04,-0.04,-0.01,0.17
75%,0.65,0.68,0.69,0.66,0.69,0.64,0.17
max,2.48,3.22,3.17,3.34,3.40,3.03,4.42


## Section 10 — Save Final Clean Dataset

Save both the fully processed train/test arrays (for direct use in Task 05)
and a single combined clean CSV (for reproducibility, sharing with
teammates, and inclusion as a Task 03 deliverable).


In [24]:
from pathlib import Path

# Notebook lives in notebooks/, so go up one level to reach project root
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)  # creates folder if it doesn't exist yet

# Save train/test splits for direct use in Task 05
X_train.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test.to_csv(PROCESSED_DIR / 'X_test.csv', index=False)
y_train.to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
y_test.to_csv(PROCESSED_DIR / 'y_test.csv', index=False)

# Save the full cleaned & engineered dataset (pre-split) for reference/report use
df_final.to_csv(PROCESSED_DIR / 'smartcare_clean_dataset.csv', index=False)

print("Files saved to:", PROCESSED_DIR)
print(" - smartcare_clean_dataset.csv  (full cleaned & engineered dataset)")
print(" - X_train.csv / X_test.csv     (scaled, encoded feature sets)")
print(" - y_train.csv / y_test.csv     (encoded target sets)")

print(f"\n=== FINAL SUMMARY ===")
print(f"Original rows:        {len(df_raw)}")
print(f"Final clean rows:     {len(df_final)}")
print(f"Rows dropped:         {len(df_raw) - len(df_final)}")
print(f"Final feature count:  {X.shape[1]}")

Files saved to: C:\Users\thrit\Desktop\smartcare-ai-project\data\processed
 - smartcare_clean_dataset.csv  (full cleaned & engineered dataset)
 - X_train.csv / X_test.csv     (scaled, encoded feature sets)
 - y_train.csv / y_test.csv     (encoded target sets)

=== FINAL SUMMARY ===
Original rows:        1000
Final clean rows:     1000
Rows dropped:         0
Final feature count:  68


## Section 11 — Data Cleaning & Feature Engineering Workflow Summary

| Step | Action Taken | Rows/Cols Affected | Rows Lost |
|---|---|---|---|
| Missing values | Split imputation: "Not Admitted" vs. mode, on `room_type` | 906 rows | 0 |
| Duplicates | Checked at 3 levels — none found | — | 0 |
| Outliers | Identified via IQR, kept (informative for target class) | ~6 clinical columns | 0 |
| Cleaning | Capped 16 rows, swapped 3 rows, documented 212 (no action) | 231 rows touched | 0 |
| Encoding | One-hot (8 nominal predictors), ordinal (target) | 8 columns | — |
| Scaling | StandardScaler, fit on X_train only | 7 numeric columns | — |
| Selection | Dropped 13 columns (identifiers, other targets, near-zero correlation) | 13 dropped | — |
| Engineering | 5 new features created (4 categorical + 1 composite score) | +5 features | — |
| Split | Stratified 80/20 train/test split | — | — |

**Result: 1000 rows retained (0 dropped), final feature set is clean, scaled,
encoded, and free of target leakage — ready for Task 05 model development.**
